# Workflow Builder - Fluent ML Pipelines

TuiML's **Workflow** builder lets you construct complete ML pipelines using a fluent, chainable API. Every method returns `self`, so you can chain preprocessing, feature engineering, training, and evaluation into a single readable expression.

**What you'll learn:**
- Build end-to-end pipelines with method chaining
- Add preprocessing steps (imputation, scaling, encoding, resampling)
- Use any preprocessor by name with `.preprocess()`
- Apply feature selection and PCA
- Configure data splits and evaluation strategies
- Export pipeline configurations as dictionaries

## 1. Basic Workflow Chain

The simplest workflow loads data, trains a model, evaluates it, and runs. Every step is a chained method call.

In [1]:
from tuiml import Workflow

# Minimal workflow: data -> train -> evaluate -> run
result = (
    Workflow("iris", target="class")
    .train("RandomForestClassifier", n_estimators=50)
    .evaluate()
    .run()
)

print(f"Model: {type(result.model).__name__}")
print(f"Metrics: {result.metrics}")

Model: RandomForestClassifier
Metrics: {'accuracy_score': 1.0, 'f1_score': 1.0}


In [2]:
# The result object holds everything: model, metrics, predictions, metadata
print(f"Predictions (first 5): {result.predictions[:5]}")
print(f"Metadata: {result.metadata}")

Predictions (first 5): [1 0 2 1 1]
Metadata: {'algorithm': 'RandomForestClassifier', 'preprocessing': [], 'feature_selection': None, 'evaluation_method': 'holdout'}


## 2. Preprocessing Chains

Chain dedicated preprocessing methods before training. Each method adds a step to the internal pipeline and returns `self`.

In [3]:
# Impute missing values with different strategies
result = (
    Workflow("diabetes", target="class")
    .impute(strategy="mean")                  # Mean imputation
    .train("NaiveBayesClassifier")
    .evaluate()
    .run()
)

print(f"Mean imputation: {result.metrics}")

Mean imputation: {'accuracy_score': 0.7647058823529411, 'f1_score': 0.6842105263157895}


In [4]:
# KNN imputation for more sophisticated missing value handling
result = (
    Workflow("diabetes", target="class")
    .impute(strategy="knn")                   # KNN imputation
    .train("NaiveBayesClassifier")
    .evaluate()
    .run()
)

print(f"KNN imputation: {result.metrics}")

KNN imputation: {'accuracy_score': 0.7647058823529411, 'f1_score': 0.6842105263157895}


In [5]:
# Normalize features to [0, 1] range
result = (
    Workflow("iris", target="class")
    .normalize(method="minmax")               # Min-max scaling
    .train("SVC", kernel="rbf")
    .evaluate()
    .run()
)

print(f"Normalized (minmax): {result.metrics}")

Normalized (minmax): {'accuracy_score': 1.0, 'f1_score': 1.0}


In [6]:
# Standardize features to zero mean, unit variance
result = (
    Workflow("iris", target="class")
    .standardize()                             # Z-score normalization
    .train("SVC", kernel="rbf")
    .evaluate()
    .run()
)

print(f"Standardized: {result.metrics}")

Standardized: {'accuracy_score': 1.0, 'f1_score': 1.0}


In [7]:
# Encode categorical variables (one-hot encoding)
result = (
    Workflow("iris", target="class")
    .encode_categorical(method="onehot")      # One-hot encoding
    .train("C45TreeClassifier")
    .evaluate()
    .run()
)

print(f"With encoding: {result.metrics}")

With encoding: {'accuracy_score': 0.3, 'f1_score': 0.4615384615384615}


In [8]:
# Chain multiple preprocessing steps together
result = (
    Workflow("diabetes", target="class")
    .impute(strategy="mean")                  # 1. Handle missing values
    .standardize()                             # 2. Z-score normalization
    .encode_categorical(method="onehot")      # 3. Encode categoricals
    .train("SVC", kernel="rbf", C=1.0)
    .evaluate()
    .run()
)

print(f"Full preprocessing chain: {result.metrics}")

Full preprocessing chain: {'accuracy_score': 0.6601307189542484, 'f1_score': 0.2121212121212121}


## 3. Any Preprocessor by Name

The `.preprocess()` method accepts any registered preprocessor class name and passes keyword arguments directly to it. This gives you access to every preprocessor in TuiML without a dedicated method.

In [9]:
# Use any preprocessor by name
result = (
    Workflow("iris", target="class")
    .preprocess("SimpleImputer", strategy="median")  # Imputer with params
    .preprocess("StandardScaler")                     # Scaler by name
    .preprocess("UselessAttributeRemover")            # Remove constant features
    .train("C45TreeClassifier")
    .evaluate()
    .run()
)

print(f"With named preprocessors: {result.metrics}")

With named preprocessors: {'accuracy_score': 1.0, 'f1_score': 1.0}


In [10]:
# Mix dedicated methods and .preprocess() freely
result = (
    Workflow("diabetes", target="class")
    .impute(strategy="mean")                          # Dedicated method
    .preprocess("MinMaxScaler")                       # By name
    .preprocess("UselessAttributeRemover")            # By name
    .train("RandomForestClassifier", n_estimators=50)
    .evaluate()
    .run()
)

print(f"Mixed methods: {result.metrics}")

Mixed methods: {'accuracy_score': 0.7254901960784313, 'f1_score': 0.6181818181818182}


## 4. Resampling for Imbalanced Data

The `.resample()` method handles class imbalance by adding a resampling step. Supported methods:
- `"smote"` -- Synthetic Minority Over-sampling Technique
- `"adasyn"` -- Adaptive Synthetic sampling
- `"random_over"` -- Random over-sampling of minority class
- `"random_under"` -- Random under-sampling of majority class

In [11]:
# SMOTE resampling for imbalanced datasets
result = (
    Workflow("diabetes", target="class")
    .impute(strategy="mean")
    .standardize()
    .resample(method="smote")                 # Balance classes with SMOTE
    .train("RandomForestClassifier", n_estimators=100)
    .evaluate()
    .run()
)

print(f"With SMOTE: {result.metrics}")

With SMOTE: {'accuracy_score': 0.785, 'f1_score': 0.7942583732057416}


In [12]:
# Random under-sampling (reduces majority class)
result = (
    Workflow("diabetes", target="class")
    .impute(strategy="mean")
    .normalize()
    .resample(method="random_under")
    .train("KNearestNeighborsClassifier", k=5)
    .evaluate()
    .run()
)

print(f"With random under-sampling: {result.metrics}")

With random under-sampling: {'accuracy_score': 0.7289719626168224, 'f1_score': 0.7289719626168225}


## 5. Feature Engineering

Add feature selection or dimensionality reduction before training.

In [13]:
# Select the top k features using SelectKBest
result = (
    Workflow("iris", target="class")
    .standardize()
    .select_features(k=2)                     # Keep top 2 features
    .train("SVC", kernel="rbf")
    .cross_validate(cv=5)
    .run()
)

print(f"With 2 best features: {result.metrics}")

With 2 best features: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.024944382578492935, 'cv_f1_score_mean': 0.9413764978982371, 'cv_f1_score_std': 0.03894739963918794}


In [14]:
# PCA dimensionality reduction (keep 2 components)
result = (
    Workflow("iris", target="class")
    .standardize()
    .pca(n_components=2)                      # Reduce to 2 dimensions
    .train("RandomForestClassifier", n_estimators=50)
    .evaluate()
    .run()
)

print(f"With PCA (2 components): {result.metrics}")

With PCA (2 components): {'accuracy_score': 0.9333333333333333, 'f1_score': 0.8888888888888888}


In [15]:
# PCA with variance threshold (keep 95% of variance)
result = (
    Workflow("glass", target="class")
    .standardize()
    .pca(n_components=0.95)                   # Keep 95% variance
    .train("NaiveBayesClassifier")
    .cross_validate(cv=5)
    .run()
)

print(f"With PCA (95% variance): {result.metrics}")

With PCA (95% variance): {'cv_accuracy_score_mean': 0.5141749723145072, 'cv_accuracy_score_std': 0.04803904508981404, 'cv_f1_score_mean': 0.26370257193786606, 'cv_f1_score_std': 0.10192861583343989}


## 6. Data Splitting

The `.split()` method configures how data is divided into train and test sets. If you don't call `.split()`, the evaluation method determines the split.

In [16]:
# Custom split: 30% test, stratified, with a fixed seed
result = (
    Workflow("iris", target="class")
    .normalize()
    .split(test_size=0.3, stratify=True, random_state=42)
    .train("RandomForestClassifier", n_estimators=100)
    .evaluate(metrics=["accuracy_score", "f1_score"])
    .run()
)

print(f"30/70 stratified split: {result.metrics}")

30/70 stratified split: {'accuracy_score': 0.9333333333333333, 'f1_score': 0.896551724137931}


In [17]:
# Larger test set for a more conservative estimate
result = (
    Workflow("glass", target="class")
    .standardize()
    .split(test_size=0.4, stratify=True, random_state=0)
    .train("KNearestNeighborsClassifier", k=5)
    .evaluate()
    .run()
)

print(f"40% test split: {result.metrics}")

40% test split: {'accuracy_score': 0.6463414634146342, 'f1_score': 0.5283018867924527}


## 7. Evaluation Strategies

TuiML supports two evaluation methods. Use `.evaluate()` for a single holdout split or `.cross_validate()` for k-fold cross-validation.

### 7.1 Holdout Evaluation

Trains on one portion and tests on the remainder. Fast, but the estimate depends on a single split.

In [18]:
# Holdout evaluation with custom metrics and test size
result = (
    Workflow("iris", target="class")
    .normalize()
    .train("RandomForestClassifier", n_estimators=100)
    .evaluate(metrics=["accuracy_score", "f1_score"], test_size=0.2)
    .run()
)

print(f"Holdout metrics: {result.metrics}")
print(f"Evaluation method: {result.metadata['evaluation_method']}")

Holdout metrics: {'accuracy_score': 1.0, 'f1_score': 1.0}
Evaluation method: holdout


### 7.2 Cross-Validation

Trains and evaluates across multiple folds. Gives a more robust performance estimate with mean and standard deviation.

In [19]:
# 10-fold cross-validation
result = (
    Workflow("iris", target="class")
    .normalize()
    .train("SVC", kernel="rbf", C=1.0)
    .cross_validate(cv=10, metrics=["accuracy_score", "f1_score"])
    .run()
)

print(f"CV metrics: {result.metrics}")
print(f"CV results: {result.cv_results}")
print(f"Evaluation method: {result.metadata['evaluation_method']}")

CV metrics: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.05333333333333332, 'cv_f1_score_mean': 0.9396153846153845, 'cv_f1_score_std': 0.08824471756153333}
CV results: {'scores': {'accuracy_score': [1.0, 1.0, 1.0, 0.9333333333333333, 1.0, 0.8666666666666667, 0.8666666666666667, 1.0, 1.0, 0.9333333333333333], 'f1_score': [1.0, 1.0, 1.0, 0.923076923076923, 1.0, 0.8, 0.75, 1.0, 1.0, 0.923076923076923]}}
Evaluation method: cross_validate


In [20]:
# 5-fold CV on a different dataset and algorithm
result = (
    Workflow("glass", target="class")
    .standardize()
    .train("KNearestNeighborsClassifier", k=3)
    .cross_validate(cv=5)
    .run()
)

print(f"Glass 5-fold CV: {result.metrics}")

Glass 5-fold CV: {'cv_accuracy_score_mean': 0.6967884828349945, 'cv_accuracy_score_std': 0.11680081228304119, 'cv_f1_score_mean': 0.6754761904761905, 'cv_f1_score_std': 0.14973067279758426}


## 8. Export Configuration

The `.to_config()` method serializes the entire workflow into a plain dictionary. This is useful for saving, sharing, or replaying pipelines.

In [23]:
# Build a workflow without running it
workflow = (
    Workflow("diabetes", target="class")
    .impute(strategy="mean")
    .standardize()
    .select_features(k=5)
    .split(test_size=0.3, stratify=True, random_state=42)
    .train("RandomForestClassifier", n_estimators=100, max_depth=10)
    .cross_validate(cv=10, metrics=["accuracy_score", "f1_score"])
)

# Export the configuration
config = workflow.to_config()

print("Exported configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

Exported configuration:
  data: {'source': 'diabetes', 'target': 'class'}
  preprocessing: [{'name': 'SimpleImputer', 'strategy': 'mean'}, {'name': 'StandardScaler'}]
  feature_selection: {'name': 'SelectKBestSelector', 'k': 5}
  split: {'test_size': 0.3, 'stratify': True, 'random_state': 42}
  model: {'name': 'RandomForestClassifier', 'params': {'n_estimators': 100, 'max_depth': 10}}
  evaluation: {'method': 'cross_validate', 'cv': 10, 'metrics': ['accuracy_score', 'f1_score']}


In [24]:
# The config is a plain dict -- you can serialize it to JSON
import json

print(json.dumps(config, indent=2))

{
  "data": {
    "source": "diabetes",
    "target": "class"
  },
  "preprocessing": [
    {
      "name": "SimpleImputer",
      "strategy": "mean"
    },
    {
      "name": "StandardScaler"
    }
  ],
  "feature_selection": {
    "name": "SelectKBestSelector",
    "k": 5
  },
  "split": {
    "test_size": 0.3,
    "stratify": true,
    "random_state": 42
  },
  "model": {
    "name": "RandomForestClassifier",
    "params": {
      "n_estimators": 100,
      "max_depth": 10
    }
  },
  "evaluation": {
    "method": "cross_validate",
    "cv": 10,
    "metrics": [
      "accuracy_score",
      "f1_score"
    ]
  }
}


In [25]:
# You can replay a config with tuiml.run()
# result = tuiml.run(config)

## 9. Complete End-to-End Pipeline

Here is a realistic pipeline that combines every capability: imputation, scaling, resampling, feature selection, training, and cross-validation -- all in a single chain.

In [26]:
from tuiml import Workflow

result = (
    Workflow("diabetes", target="class")
    .impute(strategy="mean")                          # Step 1: Fill missing values
    .standardize()                                     # Step 2: Z-score normalize
    .resample(method="smote")                          # Step 3: Balance classes
    .select_features(k=5)                              # Step 4: Keep top 5 features
    .split(test_size=0.2, stratify=True, random_state=42)  # Step 5: Configure split
    .train("RandomForestClassifier",                   # Step 6: Set algorithm
           n_estimators=100,
           max_depth=10,
           random_state=42)
    .cross_validate(cv=10,                             # Step 7: 10-fold CV
                    metrics=["accuracy_score", "f1_score"])
    .run()                                             # Step 8: Execute
)

print("=" * 55)
print("Complete Pipeline Results")
print("=" * 55)
print(f"Model:      {type(result.model).__name__}")
print(f"Metrics:    {result.metrics}")
print(f"Evaluation: {result.metadata['evaluation_method']}")
print(f"Steps:      {result.metadata['preprocessing']}")

Complete Pipeline Results
Model:      RandomForestClassifier
Metrics:    {'cv_accuracy_score_mean': 0.8039999999999999, 'cv_accuracy_score_std': 0.03231098884280701, 'cv_f1_score_mean': 0.8136044322070107, 'cv_f1_score_std': 0.02634940203951838}
Evaluation: cross_validate
Steps:      [{'name': 'SimpleImputer', 'strategy': 'mean'}, {'name': 'StandardScaler'}, {'name': 'SMOTESampler'}]


In [27]:
# Use the trained model from the result to make new predictions
import numpy as np

sample = np.array([[6, 148, 72, 35, 0, 33.6, 0.627, 50]])
prediction = result.predict(sample)
print(f"Prediction for sample: {prediction}")

Prediction for sample: [0]
